# Investigating Genes found by the Neural Networks

### NCBI Blast then Gene Search

In [ ]:
from paths import path_to_nn_runs
import os


In [ ]:
# Read nn_runs
nn_run = "inv_attr_genes_run7"
# if nn_run not in os.listdir(path_to_nn_runs):
#     raise ValueError("No run found")

def extract_kmer_line(file_path):
    # Check if filepath exists
    try:
        os.path.exists(file_path)
    except FileExistsError as e:
        print("File path doesn't exist")
    
    with open(file_path, "r") as logfile:
        for line in logfile:
            if "Top 10 decoded kmers:" in line:
                return line

def clean_kmer_line(kmer_line):
    """Clean the line containing kmers, from a messy string with noise, to a list with only decoded kmers"""
    kmers_string = kmer_line.split(":")[-1].strip()
    return kmers_string.strip("[]").replace("'", "").split(", ")

kmer_line = extract_kmer_line(path_to_nn_runs+nn_run+"/log_run7.txt")
print(clean_kmer_line(kmer_line))

In [ ]:
import time
from Bio.Blast import NCBIWWW, NCBIXML
from Bio import Entrez, SeqIO

# 1. Configuration
Entrez.email = "s215045@student.dtu.dk" 
kmers = ['CACAGCAAGCAA', 'AGAGAAGAAAGT', 'AATCACTGTCAA', 'TTCGCGTCAGAA', 'ATGACATACCAT', 'GAACAATGAGCC', 'AAGTTGAATTTG', 'ATGAAGCTGGTT', 'GGGTAAATATCC', 'AAGAGTGCTTGA']

def search_and_annotate_kmers(kmer_list, outfile:str = root+"logs/NCBI_gene_search.txt", acc_num:int = 3, tax_origin:str  = "txid38018[orgn]", ncbi_program:str = "blastn", ncbi_db:str = "core_nt"):
    """
    Blasts each of the kmers against NCBI, for related species (accessions), then searches its genes for the kmer along with possible functionalities
    """
    with open(outfile, "w") as logfile:
        print(f"Starting BLAST for {len(kmer_list)} kmers against Viral Database...", file=logfile)
        
        # We combine kmers into one FASTA-style string to save API calls
        fasta_query = "\n".join([f">kmer_{i}\n{k}" for i, k in enumerate(kmer_list)])
        
        try:
            # qblast parameters for short sequences:
            # - program: blastn
            # - database: nt (nucleotide)
            # - entrez_query: Restrict to Viruses
            # - word_size: 7 (minimum for blastn)
            # - expect: 1000 (higher to catch short hits)
            result_handle = NCBIWWW.qblast(
                program=ncbi_program, 
                database=ncbi_db, 
                sequence=fasta_query,
                entrez_query=tax_origin, #12333
                word_size=7,
                expect=1000,
                short_query=True
            )
            
            blast_records = NCBIXML.parse(result_handle)
            
            for record in blast_records:
                kmer_seq = kmer_list[int(record.query.split('_')[1])]
                print(f"\n--- Results for Kmer: {kmer_seq} ---", file=logfile)
                
                if not record.alignments:
                    print("No significant phage hits found.", file=logfile)
                    continue

                # Check the top acc_num hits for functional relevance
                for alignment in record.alignments[:acc_num]:
                    accession = alignment.accession
                    hit_def = alignment.title
                    
                    # Fetch GenBank record to find the specific gene overlapping the hit
                    print(f"Checking Gene in Hit: {accession} ({hit_def[:50]}...)", file=logfile)
                    
                    # We fetch the specific region of the hit to save bandwidth
                    hsp = alignment.hsps[0]
                    start, end = min(hsp.sbjct_start, hsp.sbjct_end), max(hsp.sbjct_start, hsp.sbjct_end)
                    
                    try:
                        handle = Entrez.efetch(db="nucleotide", id=accession, rettype="gb", retmode="text")
                        genbank_rec = SeqIO.read(handle, "genbank")
                        handle.close()
                        
                        found_gene = False
                        for feature in genbank_rec.features:
                            if feature.type == "CDS":
                                # Check if the kmer location overlaps with this gene
                                if start >= feature.location.start and end <= feature.location.end:
                                    product = feature.qualifiers.get('product', ['Unknown'])[0]
                                    gene = feature.qualifiers.get('gene', ['N/A'])[0]
                                    print(f"  [MATCH] Found in Gene: {gene} | Function: {product}", file=logfile)
                                    found_gene = True
                                    break
                        if not found_gene:
                            print("  [INFO] Hit is in an intergenic/non-coding region.", file=logfile)
                            
                    except Exception as e:
                        print(f"  [ERROR] Could not fetch details for {accession}: {e}", file=logfile)
                    
                    time.sleep(1) # Be nice to NCBI servers

        except Exception as e:
            print(f"BLAST search failed: {e}", file=logfile)

# Run the search
search_and_annotate_kmers(kmers, path_to_nn_runs+nn_run+"NCBI_gene_search.txt")

### Read and plot results file

In [ ]:
import pandas as pd
blast_results_path = path_to_nn_runs+"torch_mlp_n400_k12_standard_run92/GA_kmers_blast_results.csv"
blast_results = pd.read_csv(blast_results_path)
print(blast_results.head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,6))
sns.countplot(data=blast_results, x="Function", order=blast_results["Function"].value_counts().index)
plt.xticks(rotation=45, ha='right')
plt.title("Distribution of BLAST Hits by Organism")
plt.xlabel("Organism")
plt.ylabel("Count of Hits")
plt.tight_layout()
plt.show()

In [ ]:
organisms = sorted(blast_results["organism"].dropna().unique())
n_orgs = len(organisms)

fig, axes = plt.subplots(2, n_orgs, figsize=(7 * n_orgs, 12), sharey="row")
plt.title("Distribution of BLAST Hits by Function and Gene, per Organism", fontsize=16)

# Ensure axes is always 2D: [row][col]
if n_orgs == 1:
    axes = [[axes[0]], [axes[1]]]

for i, org in enumerate(organisms):
    # Row 1: Function
    ax_func = axes[0][i]
    subset_func = blast_results[(blast_results["organism"] == org) & (blast_results["Function"].notna())]
    order_func = subset_func["Function"].value_counts().index

    if subset_func.empty:
        ax_func.text(0.5, 0.5, "No Function data", ha="center", va="center", transform=ax_func.transAxes)
        ax_func.set_xticks([])
    else:
        sns.countplot(data=subset_func, x="Function", order=order_func, ax=ax_func)
        ax_func.tick_params(axis="x", rotation=45)
        for lbl in ax_func.get_xticklabels():
            lbl.set_ha("right")

    ax_func.set_title(f"{org} count of annotated Kmer Functions")
    ax_func.set_xlabel("Function")
    ax_func.set_ylabel("Count")

    # Row 2: Gene
    ax_gene = axes[1][i]
    subset_gene = blast_results[(blast_results["organism"] == org) & (blast_results["Gene"].notna())]
    order_gene = subset_gene["Gene"].value_counts().index

    if subset_gene.empty:
        ax_gene.text(0.5, 0.5, "No Gene data", ha="center", va="center", transform=ax_gene.transAxes)
        ax_gene.set_xticks([])
    else:
        sns.countplot(data=subset_gene, x="Gene", order=order_gene, ax=ax_gene)
        ax_gene.tick_params(axis="x", rotation=45)
        for lbl in ax_gene.get_xticklabels():
            lbl.set_ha("right")

    ax_gene.set_title(f"{org} count of annotated Kmer Genes")
    ax_gene.set_xlabel("Gene")
    ax_gene.set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
plt.subplots(figsize=(10,6), )

# Circos plot of annotated genomes

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
from pycirclize import Circos
from pycirclize.parser import Genbank
from paths import data_prod_path

def _best_feature_label(feature):
    qualifiers = feature.qualifiers
    for key in ("gene", "locus_tag", "product"):
        value = qualifiers.get(key, [""])[0] if qualifiers.get(key) else ""
        if isinstance(value, str) and value.strip():
            return value.strip()
    return ""

def _feature_midpoint(feature):
    return (int(feature.location.start) + int(feature.location.end)) / 2

def plot_circos_from_gbk(
    bact: str,
    gbk_path: str | Path | None = None,
    output_dir: str | Path | None = None,
    window_size: int = 5000,
    gene_labels: list[str] | None = None,
    label_top_n_genes: int | None = None,
    track_labels: list[str] | None = None,
    plot_sample: bool = False,
    random_seed: int = 42,
    figsize: tuple[float, float] = (10, 10),
    silent: bool = False
):
    if gbk_path is None:
        gbk_path = Path(data_prod_path) / "prokka_bacts" / bact / f"{bact}.gbk"
    else:
        gbk_path = Path(gbk_path)

    if not gbk_path.exists():
        raise FileNotFoundError(f"GenBank file not found: {gbk_path}")

    if output_dir is None:
        output_dir = Path.cwd()
    else:
        output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    gbk = Genbank(str(gbk_path))
    circos = Circos(gbk.get_seqid2size(), space=5)

    cds_plus_by_seqid = gbk.get_seqid2features(feature_type="CDS", target_strand=1)
    cds_minus_by_seqid = gbk.get_seqid2features(feature_type="CDS", target_strand=-1)
    cds_all_by_seqid = gbk.get_seqid2features(feature_type="CDS")
    trna_by_seqid = gbk.get_seqid2features(feature_type="tRNA")
    rrna_by_seqid = gbk.get_seqid2features(feature_type="rRNA")
    tmrna_by_seqid = gbk.get_seqid2features(feature_type="tmRNA")
    seq_map = gbk.get_seqid2seq()

    requested_labels = set(gene_labels) if gene_labels else set()
    labels_added = set()
    track_labels_set = set(track_labels or [])

    # Build label pool across all CDS features
    all_label_candidates = []
    seen = set()
    for seqid in cds_all_by_seqid:
        for feature in cds_all_by_seqid.get(seqid, []):
            label = _best_feature_label(feature)
            if label and label not in seen:
                all_label_candidates.append(label)
                seen.add(label)

    # Label-selection rule:
    # 1) explicit gene_labels if provided
    # 2) top N if label_top_n_genes is set
    # 3) random sampling of 1/40 labels if label_top_n_genes is None
    if requested_labels:
        selected_label_pool = list(requested_labels)
        if plot_sample:
            sample_size = max(1, len(all_label_candidates) // 40) if all_label_candidates else 0
            rng = np.random.default_rng(random_seed)
            sampled = rng.choice(all_label_candidates, size=sample_size, replace=False).tolist() if sample_size > 0 else []
            selected_label_pool.extend(sampled)
    elif label_top_n_genes is not None:
        selected_label_pool = set(all_label_candidates[:label_top_n_genes])
    else:
        sample_size = max(1, len(all_label_candidates) // 40) if all_label_candidates else 0
        rng = np.random.default_rng(random_seed)
        sampled = rng.choice(all_label_candidates, size=sample_size, replace=False).tolist() if sample_size > 0 else []
        selected_label_pool = set(sampled)

    first_sector_name = circos.sectors[0].name if circos.sectors else None

    for sector in circos.sectors:
        seqid = sector.name
        sector.axis(fc="#eeeeee", ec="none")
        center_x = (sector.start + sector.end) / 2

        # Track 1: Forward-strand CDS
        f_track = sector.add_track((92, 97), name="Forward CDS")
        f_feats = cds_plus_by_seqid.get(seqid, [])
        if f_feats:
            f_track.genomic_features(f_feats, color="salmon", lw=0.3)

        # Track 2: Reverse-strand CDS
        r_track = sector.add_track((87, 92), name="Reverse CDS")
        r_feats = cds_minus_by_seqid.get(seqid, [])
        if r_feats:
            r_track.genomic_features(r_feats, color="skyblue", lw=0.3)

        # Track 3: RNA features
        rna_track = sector.add_track((82, 85), name="RNAs")
        rna_feats = [
            *trna_by_seqid.get(seqid, []),
            *rrna_by_seqid.get(seqid, []),
            *tmrna_by_seqid.get(seqid, []),
        ]
        if rna_feats:
            rna_track.genomic_features(rna_feats, color="black", lw=1)

        # Track 4: GC content
        gc_track = sector.add_track((68, 78), name="GC Content")
        seq_text = seq_map.get(seqid, "")
        pos_gc, content = gbk.calc_gc_content(window_size=window_size, seq=seq_text)
        gc_min = float(np.min(content)) if len(content) else 0.0
        gc_max = float(np.max(content)) if len(content) else 1.0
        gc_mid = (gc_min + gc_max) / 2.0
        gc_hi = np.where(content > gc_mid, content, gc_mid)
        gc_lo = np.where(content < gc_mid, content, gc_mid)
        gc_track.line(pos_gc, content, color="black", lw=0.5)
        gc_track.fill_between(pos_gc, gc_hi, gc_mid, vmin=gc_min, vmax=gc_max, color="green", alpha=0.4)
        gc_track.fill_between(pos_gc, gc_lo, gc_mid, vmin=gc_min, vmax=gc_max, color="red", alpha=0.4)

        # Track 5: GC skew
        skew_track = sector.add_track((56, 66), name="GC Skew")
        pos_skew, skew = gbk.calc_gc_skew(window_size=window_size, seq=seq_text)
        max_abs_skew = float(np.max(np.abs(skew))) if len(skew) else 1.0
        skew_pos = np.where(skew > 0, skew, 0.0)
        skew_neg = np.where(skew < 0, skew, 0.0)
        skew_track.fill_between(
            pos_skew,
            skew_pos,
            0.0,
            vmin=-max_abs_skew,
            vmax=max_abs_skew,
            color="purple",
            alpha=0.4,
        )
        skew_track.fill_between(
            pos_skew,
            skew_neg,
            0.0,
            vmin=-max_abs_skew,
            vmax=max_abs_skew,
            color="orange",
            alpha=0.4,
        )

        # Outer label track: keep all text outside plotted data tracks
        outer_label_track = sector.add_track((104, 126), name="Outer Labels")

        # Gene labels/markings for identification (outside tracks)
        cds_feats = cds_all_by_seqid.get(seqid, [])
        label_candidates = []
        for feature in cds_feats:
            label = _best_feature_label(feature)
            if not label or label in labels_added:
                continue
            if label in selected_label_pool:
                label_candidates.append((feature, label))

        for feature, label in label_candidates:
            midpoint = _feature_midpoint(feature)
            outer_label_track.annotate(
                midpoint,
                label,
                min_r=105,
                max_r=124,
                label_size=7,
                shorten=24,
                line_kws={"color": "dimgray", "lw": 0.7},
                text_kws={"color": "black"},
            )
            labels_added.add(label)

        # Optional track-name labels, also outside tracks
        if seqid == first_sector_name:
            if "forward_cds" in track_labels_set:
                outer_label_track.text("Forward CDS", x=center_x, r=118, size=8, color="salmon")
            if "reverse_cds" in track_labels_set:
                outer_label_track.text("Reverse CDS", x=center_x, r=116, size=8, color="deepskyblue")
            if "rna" in track_labels_set:
                outer_label_track.text("RNA", x=center_x, r=114, size=8, color="black")
            if "gene_labels" in track_labels_set:
                outer_label_track.text("Gene labels", x=center_x, r=112, size=8, color="dimgray")
            if "gc_content" in track_labels_set:
                outer_label_track.text("GC content", x=center_x, r=110, size=8, color="black")
            if "gc_skew" in track_labels_set:
                outer_label_track.text("GC skew", x=center_x, r=108, size=8, color="purple")

    fig = circos.plotfig(figsize=figsize)
    fig.suptitle(f"Circos Genome Map: {bact}", fontsize=14, y=0.98)

    legend_handles = [
        mpatches.Patch(color="salmon", label="CDS (+ strand)"),
        mpatches.Patch(color="skyblue", label="CDS (- strand)"),
        mpatches.Patch(color="black", label="RNA genes (tRNA/rRNA/tmRNA)"),
        mlines.Line2D([], [], color="dimgray", lw=1, label="Annotated gene label marker"),
        mpatches.Patch(color="green", alpha=0.5, label="GC content above midpoint"),
        mpatches.Patch(color="red", alpha=0.5, label="GC content below midpoint"),
        mpatches.Patch(color="purple", alpha=0.5, label="GC skew positive"),
        mpatches.Patch(color="orange", alpha=0.5, label="GC skew negative"),
    ]
    ax = fig.axes[0]
    ax.legend(
        handles=legend_handles,
        loc="upper right",
        bbox_to_anchor=(0, 0),
        fontsize=8,
        frameon=False,
    )

    output_png = output_dir / f"{bact}_genome_map.png"
    fig.savefig(output_png, dpi=300, bbox_inches="tight")
    if not silent: print(f"Saved Circos plot to: {output_png}")

    return fig, output_png

# Example run
bact = "J105_22_reoriented_merged"
fig, output_png = plot_circos_from_gbk(
    bact = bact,
    output_dir = f"{Path(data_prod_path)}/prokka_bacts/{bact}/",
    gene_labels = [
    "cas1", "cas2-3", "csy1", "csy2", "csy3", "csy4", 
    "mazE", "mazF", "relB", "relE", "hicA", "hicB", 
    "hipA", "hipB", "parE", "hsdS", "hsdM", "hsdR", 
    "dndC", "dndD"],
    plot_sample=True, 
    silent = True
)

### In batch for all annotated bacterial genomes

In [ ]:
import os
from tqdm import tqdm
for dir in tqdm(os.listdir(data_prod_path+"/prokka_bacts/")):
    bact = dir.split("_reoriented_merged")[0]
    inner_dir = os.path.join(data_prod_path, "prokka_bacts", dir)
    for file in os.listdir(inner_dir):
        if file.endswith(".gbk"):
            gbk_path = os.path.join(inner_dir, file)
            output_dir = inner_dir
            #print(f"Processing {bact}...")
            #print(f"GBK path: {gbk_path}")
            plot_circos_from_gbk(
                bact=bact,
                gbk_path=gbk_path,
                output_dir=output_dir,
                gene_labels=[
                    "cas1", "cas2-3", "csy1", "csy2", "csy3", "csy4", 
                    "mazE", "mazF", "relB", "relE", "hicA", "hicB", 
                    "hipA", "hipB", "parE", "hsdS", "hsdM", "hsdR", 
                    "dndC", "dndD"],
                plot_sample=True
            )

# Mapping PFI values to Kmers in Genes
1) read and locate high/low pfi value hash values
2) decode using KmerCodecs (not available for sourmash)


In [ ]:
from pathlib import Path
import pandas as pd
from Bio import SeqIO
from paths import data_prod_path
from manipulations import clean_bact_names

ANNOTATED_SNIPPET_ROOT = Path(data_prod_path) / "annotated_snippet"

def _normalize_kmer(kmer: str) -> str:
    return str(kmer).strip().upper()

def extract_bacteria_genes_for_kmer(kmer: str, strain_name: str, root_dir: str | Path = ANNOTATED_SNIPPET_ROOT) -> pd.DataFrame:
    """Return bacterial gene annotations for records whose sequence contains `kmer`.

    Searches for:
    - a file ending in `_merged.ffn`
    - a companion file ending in `_merged.tsv`

    The matching record IDs from the FASTA headers are matched against the
    `locus_tag` column in the TSV file.
    """
    root_dir = Path(root_dir)
    kmer = _normalize_kmer(kmer)

    strain_dirs = [p for p in (root_dir / "prokka_bacts").rglob("*") if p.is_dir() and strain_name in p.name]
    if not strain_dirs:
        raise FileNotFoundError(f"No bacteria directory found for strain '{strain_name}' under {root_dir / 'prokka_bacts'}")

    strain_dir = strain_dirs[0]
    ffn_files = sorted(strain_dir.glob("*_merged.ffn"))
    tsv_files = sorted(strain_dir.glob("*_merged.tsv"))
    if not ffn_files:
        raise FileNotFoundError(f"No *_merged.ffn file found in {strain_dir}")
    if not tsv_files:
        raise FileNotFoundError(f"No *_merged.tsv file found in {strain_dir}")

    matching_locus_tags = []
    for record in SeqIO.parse(str(ffn_files[0]), "fasta"):
        if kmer in str(record.seq).upper():
            matching_locus_tags.append(record.id)

    cols = ["bact", "locus_tag", "kmer_in_seq", "length_bp", "gene", "product"]
    if not matching_locus_tags:
        return pd.DataFrame(columns=cols)

    ann_df = pd.read_csv(tsv_files[0], sep="\t")
    ann_df["kmer_in_seq"] = kmer
    ann_df["bact"] = clean_bact_names(strain_name)
    missing = [c for c in cols if c not in ann_df.columns]
    if missing:
        raise KeyError(f"Missing expected columns in {tsv_files[0]}: {missing}")

    result = ann_df[ann_df["locus_tag"].astype(str).isin(matching_locus_tags)][cols].copy()
    return result.reset_index(drop=True)

def extract_phage_genes_for_kmer(kmer: str, strain_name: str, root_dir: str | Path = ANNOTATED_SNIPPET_ROOT) -> pd.DataFrame:
    """Return phage gene annotations for records whose sequence contains `kmer`.

    Searches for:
    - a `phanotate.ffn` file under the pharokka results for the strain
    - a `*_per_cds_predictions.tsv` file under the phold results for the strain

    The matching FASTA record IDs are matched against the `cds_id` column in the
    PHOLD table.
    """
    root_dir = Path(root_dir)
    kmer = _normalize_kmer(kmer)

    pharokka_root = root_dir / "pharokka"
    phold_root = root_dir / "phold"

    pharokka_dirs = [p for p in pharokka_root.rglob("*") if p.is_dir() and strain_name in p.name]
    phold_dirs = [p for p in phold_root.rglob("*") if p.is_dir() and strain_name in p.name]
    if not pharokka_dirs:
        raise FileNotFoundError(f"No pharokka directory found for strain '{strain_name}' under {pharokka_root}")
    if not phold_dirs:
        raise FileNotFoundError(f"No phold directory found for strain '{strain_name}' under {phold_root}")

    pharokka_dir = pharokka_dirs[0]
    phold_dir = phold_dirs[0]

    ffn_files = sorted(pharokka_dir.glob("**/phanotate.ffn"))
    if not ffn_files:
        raise FileNotFoundError(f"No phanotate.ffn file found in {pharokka_dir}")

    tsv_files = sorted(phold_dir.glob("**/*_per_cds_predictions.tsv"))
    if not tsv_files:
        raise FileNotFoundError(f"No *_per_cds_predictions.tsv file found in {phold_dir}")

    matching_cds_ids = []
    for record in SeqIO.parse(str(ffn_files[0]), "fasta"):
        if kmer in str(record.seq).upper():
            matching_cds_ids.append(record.id)

    cols = [
        "contig_id", "cds_id", "kmer_in_seq", "start", "end", "phrog", "function", "product",
        "annotation_method", "annotation_confidence", "tophit_protein",
        "function_with_highest_bitscore_proportion", "prostt5_confidence"
    ]

    if not matching_cds_ids:
        return pd.DataFrame(columns=cols)

    ann_df = pd.read_csv(tsv_files[0], sep="\t")
    ann_df["kmer_in_seq"] = kmer
    missing = [c for c in cols if c not in ann_df.columns]
    if missing:
        raise KeyError(f"Missing expected columns in {tsv_files[0]}: {missing}")

    result = ann_df[ann_df["cds_id"].astype(str).isin(matching_cds_ids)][cols].copy()
    return result.reset_index(drop=True)


In [2]:
extract_bacteria_genes_for_kmer("CACAGCAAGCAA", "J76_21_reoriented_merged")

,bact,locus_tag,kmer_in_seq,length_bp,gene,product
0,J76_21,AHHNCAAL_03872,CACAGCAAGCAA,390,tusD,NaN
1,J76_21,AHHNCAAL_03872,CACAGCAAGCAA,390,tusD,Sulfurtransferase TusD


In [3]:
extract_phage_genes_for_kmer("CACAA", "Lelliottia_phage_Pantea")

,contig_id,cds_id,kmer_in_seq,start,end,phrog,function,product,annotation_method,annotation_confidence,tophit_protein,function_with_highest_bitscore_proportion,prostt5_confidence
0,Lelliottia_phage_Pantea,QEWVWKBE_CDS_0001,CACAA,186,1,No_PHROG,unknown function,hypothetical protein,none,none,NaN,NaN,51.90625
1,Lelliottia_phage_Pantea,QEWVWKBE_CDS_0004,CACAA,686,414,No_PHROG,unknown function,hypothetical protein,none,none,NaN,NaN,45.37500
2,Lelliottia_phage_Pantea,QEWVWKBE_CDS_0008,CACAA,1674,1387,30110,unknown function,hypothetical protein,foldseek,high,envhog_8JHlt,unknown function,62.37500
3,Lelliottia_phage_Pantea,QEWVWKBE_CDS_0010,CACAA,2115,1933,30457,unknown function,hypothetical protein,foldseek,high,protein104025,unknown function,81.25000
4,Lelliottia_phage_Pantea,QEWVWKBE_CDS_0012,CACAA,2782,2354,14785,unknown function,hypothetical protein,foldseek,high,protein381140,unknown function,43.90625
...,...,...,...,...,...,...,...,...,...,...,...,...,...
129,Lelliottia_phage_Pantea,QEWVWKBE_CDS_0275,CACAA,143725,143961,No_PHROG,unknown function,hypothetical protein,none,none,NaN,NaN,50.96875
130,Lelliottia_phage_Pantea,QEWVWKBE_CDS_0279,CACAA,144591,144932,7366,unknown function,hypothetical protein,pharokka,pharokka,NaN,NaN,36.43750
131,Lelliottia_phage_Pantea,QEWVWKBE_CDS_0292,CACAA,148417,148782,60005,unknown function,hypothetical protein,foldseek,high,singleton27042,unknown function,43.75000
132,Lelliottia_phage_Pantea,QEWVWKBE_CDS_0294,CACAA,149078,149218,No_PHROG,unknown function,hypothetical protein,none,none,NaN,NaN,48.81250


### Extract multiple entries

In [ ]:
### Bacteria ###
def batch_bact_annotate(bkmers : list, bact_names : list, TS : bool = False) -> pd.DataFrame:
    bact_annotations = pd.DataFrame(columns=["bact", "locus_tag", "kmer_in_seq", "length_bp", "gene", "product"])
    for bact in bact_names:
        for kmer in bkmers:
            if TS: print(f"\n=== Bacterial genes for bact-kmer: {bact}-{kmer} ===")
            try:
                bact_genes = extract_bacteria_genes_for_kmer(kmer, bact)
                if bact_genes.empty:
                    if TS: print("No bacterial genes found containing this kmer.")
                else:
                    bact_annotations = pd.concat([bact_annotations, bact_genes], ignore_index=True)
            except Exception as e:
                if TS: print(f"Error extracting bacterial genes for kmer '{kmer}': {e}")

    return bact_annotations

### Phages ###
def batch_phage_annotate(pkmers : list, phage_names : list, TS : bool = False) -> pd.DataFrame:
    phage_annotations = pd.DataFrame(columns=[
            "contig_id", "cds_id", "kmer_in_seq", "start", "end", "phrog", "function", "product",
            "annotation_method", "annotation_confidence", "tophit_protein",
            "function_with_highest_bitscore_proportion", "prostt5_confidence"
        ])
    for phage in phage_names:
        for kmer in pkmers:
            if TS: print(f"\n=== Phage genes for phage-kmer: {phage}-{kmer} ===")
            try:
                phage_genes = extract_phage_genes_for_kmer(kmer, phage)
                if phage_genes.empty:
                    if TS: print("No phage genes found containing this kmer.")
                else:
                    phage_annotations = pd.concat([phage_annotations, phage_genes], ignore_index=True)
            except Exception as e:
                if TS: print(f"Error extracting phage genes for kmer '{kmer}': {e}")

    return phage_annotations

In [5]:
bkmers = ['CACAGCAAGCAA', 'AGAGAAGAAAGT', 'AATCACTGTCAA', 'TTCGCGTCAGAA', 'ATGACATACCAT', 'GAACAATGAGCC', 'AAGTTGAATTTG', 'ATGAAGCTGGTT', 'GGGTAAATATCC', 'AAGAGTGCTTGA']
bact_names = ["J76_21_reoriented_merged", "J62_22_reoriented_merged"]
display(batch_bact_annotate(bkmers, bact_names, TS=True))

phage_names = ["Lelliottia_phage_Pantea", "Pectobacterium_phage_Amona"]
pkmers = ['CACAA', 'AGAT', 'AATCCAA', 'TTAA', 'ATGAT', 'GAACAATGAGCC', 'AAGTTGAATTTG', 'ATGGTT', 'GGGTAAATATCC', 'AAGAGA']
display(batch_phage_annotate(pkmers, phage_names, TS=True))


=== Bacterial genes for bact-kmer: J76_21_reoriented_merged-CACAGCAAGCAA ===

=== Bacterial genes for bact-kmer: J76_21_reoriented_merged-AGAGAAGAAAGT ===
No bacterial genes found containing this kmer.

=== Bacterial genes for bact-kmer: J76_21_reoriented_merged-AATCACTGTCAA ===
No bacterial genes found containing this kmer.

=== Bacterial genes for bact-kmer: J76_21_reoriented_merged-TTCGCGTCAGAA ===
No bacterial genes found containing this kmer.

=== Bacterial genes for bact-kmer: J76_21_reoriented_merged-ATGACATACCAT ===
No bacterial genes found containing this kmer.

=== Bacterial genes for bact-kmer: J76_21_reoriented_merged-GAACAATGAGCC ===
No bacterial genes found containing this kmer.

=== Bacterial genes for bact-kmer: J76_21_reoriented_merged-AAGTTGAATTTG ===

=== Bacterial genes for bact-kmer: J76_21_reoriented_merged-ATGAAGCTGGTT ===
No bacterial genes found containing this kmer.

=== Bacterial genes for bact-kmer: J76_21_reoriented_merged-GGGTAAATATCC ===

=== Bacterial g

,bact,locus_tag,kmer_in_seq,length_bp,gene,product
0,J76_21,AHHNCAAL_03872,CACAGCAAGCAA,390,tusD,NaN
1,J76_21,AHHNCAAL_03872,CACAGCAAGCAA,390,tusD,Sulfurtransferase TusD
2,J76_21,AHHNCAAL_01791,AAGTTGAATTTG,750,artP,NaN
3,J76_21,AHHNCAAL_01791,AAGTTGAATTTG,750,artP,Arginine transport ATP-binding protein ArtP
4,J76_21,AHHNCAAL_02831,GGGTAAATATCC,543,nuoI,NaN
5,J76_21,AHHNCAAL_02831,GGGTAAATATCC,543,nuoI,NADH-quinone oxidoreductase subunit I



=== Phage genes for phage-kmer: Lelliottia_phage_Pantea-CACAA ===

=== Phage genes for phage-kmer: Lelliottia_phage_Pantea-AGAT ===

=== Phage genes for phage-kmer: Lelliottia_phage_Pantea-AATCCAA ===

=== Phage genes for phage-kmer: Lelliottia_phage_Pantea-TTAA ===

=== Phage genes for phage-kmer: Lelliottia_phage_Pantea-ATGAT ===

=== Phage genes for phage-kmer: Lelliottia_phage_Pantea-GAACAATGAGCC ===
No phage genes found containing this kmer.

=== Phage genes for phage-kmer: Lelliottia_phage_Pantea-AAGTTGAATTTG ===
No phage genes found containing this kmer.

=== Phage genes for phage-kmer: Lelliottia_phage_Pantea-ATGGTT ===

=== Phage genes for phage-kmer: Lelliottia_phage_Pantea-GGGTAAATATCC ===
No phage genes found containing this kmer.

=== Phage genes for phage-kmer: Lelliottia_phage_Pantea-AAGAGA ===

=== Phage genes for phage-kmer: Pectobacterium_phage_Amona-CACAA ===
Error extracting phage genes for kmer 'CACAA': No pharokka directory found for strain 'Pectobacterium_phage_

,contig_id,cds_id,kmer_in_seq,start,end,phrog,function,product,annotation_method,annotation_confidence,tophit_protein,function_with_highest_bitscore_proportion,prostt5_confidence
0,Lelliottia_phage_Pantea,QEWVWKBE_CDS_0001,CACAA,186,1,No_PHROG,unknown function,hypothetical protein,none,none,NaN,NaN,51.90625
1,Lelliottia_phage_Pantea,QEWVWKBE_CDS_0004,CACAA,686,414,No_PHROG,unknown function,hypothetical protein,none,none,NaN,NaN,45.375
2,Lelliottia_phage_Pantea,QEWVWKBE_CDS_0008,CACAA,1674,1387,30110,unknown function,hypothetical protein,foldseek,high,envhog_8JHlt,unknown function,62.375
3,Lelliottia_phage_Pantea,QEWVWKBE_CDS_0010,CACAA,2115,1933,30457,unknown function,hypothetical protein,foldseek,high,protein104025,unknown function,81.25
4,Lelliottia_phage_Pantea,QEWVWKBE_CDS_0012,CACAA,2782,2354,14785,unknown function,hypothetical protein,foldseek,high,protein381140,unknown function,43.90625
...,...,...,...,...,...,...,...,...,...,...,...,...,...
856,Lelliottia_phage_Pantea,QEWVWKBE_CDS_0257,AAGAGA,137399,138223,12959,unknown function,hypothetical protein,foldseek,high,protein30708,unknown function,51.625
857,Lelliottia_phage_Pantea,QEWVWKBE_CDS_0269,AAGAGA,141792,142358,4273,unknown function,hypothetical protein,foldseek,high,protein81706,unknown function,45.8125
858,Lelliottia_phage_Pantea,QEWVWKBE_CDS_0281,AAGAGA,145222,145554,14450,unknown function,hypothetical protein,foldseek,high,protein189414,unknown function,46.75
859,Lelliottia_phage_Pantea,QEWVWKBE_CDS_0286,AAGAGA,146672,147148,20562,unknown function,hypothetical protein,pharokka,pharokka,NaN,NaN,44.46875


### Try with class from analysis.py

In [2]:
from analysis import GeneAnalysis
from paths import data_prod_path
bkmers = ['CACAGCAAGCAA', 'AGAGAAGAAAGT', 'AATCACTGTCAA', 'TTCGCGTCAGAA', 'ATGACATACCAT', 'GAACAATGAGCC', 'AAGTTGAATTTG', 'ATGAAGCTGGTT', 'GGGTAAATATCC', 'AAGAGTGCTTGA']
bact_names = ["J76_21", "J62_22"]
phage_names = ["Lelliottia_phage_Pantea", "Pectobacterium_phage_Amona"]
pkmers = ['CACAA', 'AGAT', 'AATCCAA', 'TTAA', 'ATGAT', 'GAACAATGAGCC', 'AAGTTGAATTTG', 'ATGGTT', 'GGGTAAATATCC', 'AAGAGA']

GA = GeneAnalysis()
#GA.batch_bact_annotate(bkmers, bact_names, data_prod_path)
GA.batch_phage_annotate(pkmers, phage_names, data_prod_path)

Annotating phage-kmer pairs: 100%|██████████| 20/20 [00:02<00:00,  7.29it/s]


,contig_id,cds_id,kmer_in_seq,start,end,phrog,function,product,annotation_method,annotation_confidence,tophit_protein,function_with_highest_bitscore_proportion,prostt5_confidence
0,Lelliottia_phage_Pantea,QEWVWKBE_CDS_0001,CACAA,186,1,No_PHROG,unknown function,hypothetical protein,none,none,NaN,NaN,51.90625
1,Lelliottia_phage_Pantea,QEWVWKBE_CDS_0004,CACAA,686,414,No_PHROG,unknown function,hypothetical protein,none,none,NaN,NaN,45.375
2,Lelliottia_phage_Pantea,QEWVWKBE_CDS_0008,CACAA,1674,1387,30110,unknown function,hypothetical protein,foldseek,high,envhog_8JHlt,unknown function,62.375
3,Lelliottia_phage_Pantea,QEWVWKBE_CDS_0010,CACAA,2115,1933,30457,unknown function,hypothetical protein,foldseek,high,protein104025,unknown function,81.25
4,Lelliottia_phage_Pantea,QEWVWKBE_CDS_0012,CACAA,2782,2354,14785,unknown function,hypothetical protein,foldseek,high,protein381140,unknown function,43.90625
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1078,Pectobacterium_phage_Amona,TCKYBVKA_CDS_0026,AAGAGA,13646,15520,24,tail,tail length tape measure protein,foldseek,high,protein541794,tail,59.4375
1079,Pectobacterium_phage_Amona,TCKYBVKA_CDS_0028,AAGAGA,15948,16520,No_PHROG,unknown function,hypothetical protein,none,none,NaN,NaN,70.75
1080,Pectobacterium_phage_Amona,TCKYBVKA_CDS_0043,AAGAGA,28955,27960,412,"DNA, RNA and nucleotide metabolism",exonuclease VIII,foldseek,high,protein721464,"DNA, RNA and nucleotide metabolism",67.0625
1081,Pectobacterium_phage_Amona,TCKYBVKA_CDS_0045,AAGAGA,29774,29319,824,"DNA, RNA and nucleotide metabolism",nuclease,foldseek,high,protein862828,"DNA, RNA and nucleotide metabolism",62.0


# Testing GAPlottingUtils - collect_iterres

In [ ]:
from collect_iterres import *
import os
import re
import ast
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import networkx as nx
import seaborn as sns
import numpy as np
from tqdm import tqdm
import argparse
from decimal import Decimal
from time import time, sleep
from datetime import datetime
from paths import data_prod_path, path_to_nn_runs
from analysis import GeneAnalysis, PFI_Lookup
import json
import joblib
from analysis import regain_kmers

base_dir = path_to_nn_runs+"IterExcl_encoded_sketches_n500_k12"
outdir = data_prod_path+"/GAplot_test_n500_k12/"
x_col = None
hue_col = None
group_x_col = None
group_hue_col = None
weight_pfi = False
filter_harsh = False
top_kmers = 200
network_top_kmers = 200


/net/domus/home/people/s215045/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Defnining main

In [ ]:
all_data = []
top_kmers_df = pd.DataFrame() # Placeholder top_kmers_csv file

if not os.path.exists(base_dir):
    logger.log(f"Directory {base_dir} not found.")
else:
    logger.log(f"Scanning directory: {base_dir}")

if not os.path.exists(outdir):
    os.makedirs(outdir, exist_ok=True)
    logger.log(f"Created output directory: {outdir}")
else:
    logger.log(f"Output directory already exists: {outdir}")

logfile_path = os.path.join(outdir, "collect_iterres_log.txt")
logfile = open(logfile_path, 'w')
logger.set_logfile(logfile)
logger.log(f"{datetime.now().strftime('[%Y-%m-%d %H:%M:%S] ')} collect_iterres started. Scanning {base_dir} for log files.")

# Iterate through all folders in nn_runs
c = 0
for folder_name in tqdm(os.listdir(base_dir), desc="Processing folders"):
    logger.log(f"\n[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Processing folder: {folder_name}")
    folder_path = os.path.join(base_dir, folder_name)
    if os.path.isdir(folder_path):
        top_int_kmer_success = False
        
        # Search for log files in this specific run folder
        for file in os.listdir(folder_path):
            # Find failed runs and sort them

            # Extract metrics from log files 
            if file.endswith(".txt") and "log_run" in file.lower():
                log_path = os.path.join(folder_path, file)
                metrics, run_info = extract_metrics_from_log(log_path)
                metrics['folder'] = folder_name
                all_data.append(metrics)
                #print("run_info", run_info)

                if True:
                    #print("hk_lookup not provided. Attempting to deduce it from log info for encoded data2 run.")
                    # If hk_lookup is not provided, try to deduce it from the log info
                    try:
                        dir = "encoded_sketches" if run_info['use_encoded'] else "SM_sketches"
                        if run_info['data2']:
                            dir += "_data2"

                        hk_path = os.path.join(data_prod_path, dir, f"hk_lookup_n{metrics['n']}_k{metrics['k']}.json")
                        kmer_to_gene = open_hk_lookup(hk_path, reverse=True)
                        if kmer_to_gene is not None:
                            logger.log(f"Deduced hk_lookup from log info for {folder_name} using path: {hk_path}")
                    except Exception as e:
                        print(f"Error deducing hk_lookup from log info in {log_path}: {e}")

            # Extract top kmers from pair_kmers.csv files
            elif file.endswith("pair_kmers.csv"):
                logger.log(f"Found top kmers file: {file} in folder: {folder_name}")
                top_kmers_path = os.path.join(folder_path, file)
                try:
                    df_kmers = pd.read_csv(top_kmers_path)
                    df_kmers['folder'] = folder_name
                    df_kmers["UPS"] = calculate_unified_score(metrics)
                    top_int_kmer_success = True
                except Exception as e:
                    logger.log(f"Error reading {top_kmers_path}: {e}")
            
            elif file.startswith("pfi_objects") and os.path.isdir(file):
                logger.log(f"Found PFI objects directory: {file} in folder: {folder_name}")
                pfi_path = os.path.join(folder_path, file)
                try:
                    interaction_pairs = joblib.load(pfi_path + "interaction_pairs.jbl")
                    occurence_pairs = joblib.load(pfi_path + "occurence_pairs.jbl")
                    interaction_freq_pairs = joblib.load(pfi_path + "interaction_freq_pairs.jbl")
                    occurence_freq_pairs = joblib.load(pfi_path + "occurence_freq_pairs.jbl")
                    expected_interactions = joblib.load(pfi_path + "expected_interactions.jbl")
                    top_pairs = sorted(interaction_pairs.items(), key=lambda x: expected_interactions.get(x[0], 0), reverse=True)[:top_kmers]
                except Exception as e:
                    logger.log(f"Error loading PFI lookup from {pfi_path}: {e}")
        
        else:
            logger.log(f"Skipping PFI calculation for {folder_name}. Reason: top_kmers={top_int_kmer_success}, hk_lookup={kmer_to_gene is not None}")
        
        if top_int_kmer_success:
            top_kmers_df = pd.concat([top_kmers_df, df_kmers], ignore_index=True)
    logger.log(f"\n[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Folder processed: {folder_name}")
    logger.log("#" * 50)
    c += 1
    if c > 4:
        break

### Sorting top_kmers_df by weighted PFI score (if weight_pfi flag is set)
if not top_kmers_df.empty:
    top_kmers_go = True
    if weight_pfi and "UPS" in top_kmers_df.columns:
        top_kmers_df = top_kmers_df.sort_values(by="UPS", ascending=False)
        logger.log("Sorted top_kmers_df by Unified Performance Score (UPS).")
    else:
        logger.log("Warning: 'UPS' column not found in top_kmers_df. Skipping sorting by UPS.")
else:
    logger.log("top_kmers_df is empty. No k-mer data to process or plot.")
    top_kmers_go = False



### Metrics Extraction Summary and Plotting ###
if all_data:
    df = pd.DataFrame(all_data)
    logger.log(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Extracted metrics from {len(df)} log files.")

    #check if any values of the cols in below_one_cols are above 1, if so, apply the correct_deci_number function to the entire column
    try:
        below_one_cols = ['test_accuracy', 'test_balanced_accuracy', 'unseen_test_accuracy', 'unseen_test_balanced_accuracy', 'precision', 'recall', 'f1']
        for col in below_one_cols:
            if col in df.columns:
                if (df[col] > 1).any():
                    logger.log(f"Column '{col}' contains values greater than 1. Applying correction to entire column.")
                    df[col] = df[col].apply(correct_deci_number)
                else:
                    continue
            else:
                logger.log(f"Column '{col}' not found in dataframe. Skipping correction for this column.")
    except Exception as e:
        logger.log(f"Error during decimal correction: {e}")

    try:
        group_x_col = group_x_col.lower()
    except Exception as e:
        logger.log(f"Unable to process group_x_col: {group_x_col}, error: {e}. Defaulting to no grouping.")

    # Marking TP = 0 runs as failed runs for better visualization in the confusion matrix bar plot and heatmap
    if 'TP' in df.columns:
        df['status'] = df.apply(lambda row: False if row['TP'] == 0 or row['TP'] is None else row['status'], axis=1)
    
    if filter_harsh:
        prec_recall_threshold = 0.5
        if len(df["precision"].dropna()) > 0:
            if (df["precision"] > 0.5).any() and (df["recall"] > prec_recall_threshold).any():
                df['status'] = df.apply(lambda row: False if row['precision'] <= prec_recall_threshold or row['precision'] is None else row['status'], axis=1)
                df['status'] = df.apply(lambda row: False if row['recall'] <= prec_recall_threshold or row['recall'] is None else row['status'], axis=1)
        if len(df["test_accuracy"].dropna()) > 0:
            if (df["test_accuracy"] > 0.5).any():
                df['status'] = df.apply(lambda row: False if row['test_accuracy'] <= 0.5 or row['test_accuracy'] is None else row['status'], axis=1)
        
    
    # Subsetting df to only include successful runs
    if False in df["status"].values:
        logger.log(f"⚠ WARNING: Some runs have failed! Count: {(df['status'] == False).sum()}")
        
        #Get the list of failed runs folder names            
        failed_runs = df[df['status'] == False]['folder'].tolist()
        df_all = df.copy()
        if (df['status'] == True).any():
            df = df[df['status'] == True]
        else:
            raise ValueError("All runs have failed. No data to plot.")
        
        #Subset the top_kmers_df to only include the successful runs as well
        if top_kmers_go:
            logger.log(top_kmers_df.head().to_string())
            top_kmers_df = top_kmers_df[~top_kmers_df['folder'].isin(failed_runs)]
            logger.log(f"Subsetted dataframe to {len(df)} successful runs for plotting. Also subsetted top_kmers_df to {len(top_kmers_df)} entries corresponding to successful runs.")

        # Obtain b_value and p_value from each failed run
        try: 
            failed_runs_info = []
            for folder in failed_runs:
                b_value = None
                p_value = None
                try:
                    b_match = re.search(r"b(\d+)", folder)
                    p_match = re.search(r"p(\d+)", folder)
                    if b_match:
                        b_value = b_match.group(1)
                    if p_match:
                        p_value = p_match.group(1)
                except Exception as e:
                    logger.log(f"Error extracting b_value and p_value from folder name '{folder}': {e}")
                failed_runs_info.append((folder, b_value, p_value))
            logger.log("Failed runs and their corresponding b_value and p_value:")
            for folder, b_value, p_value in failed_runs_info:
                logger.log(f"  {folder}: b={b_value}, p={p_value}")
        except Exception as e:
            logger.log(f"Error processing failed runs for b_value and p_value extraction: {e}")
    
    plotting = MetricPlottingUtils(df=df, outdir=str(outdir), x_col=x_col, hue_col=hue_col, x_col_by_cluster=(group_x_col == 'cluster'), x_col_by_phage=(group_x_col == 'phage'))
    #plotting.plot_graphs()
    # Save the raw data for inspection
    df.to_csv(outdir +'all_runs_summary.csv', index=False)
    if top_kmers_go:
        top_kmers_df.to_csv(outdir + 'top_kmers_summary.csv', index=False)
    logger.log("✓ Summary CSVs saved as all_runs_summary.csv and top_kmers_summary.csv")
else:
    logger.log("No valid data found for Metrics Plotting.")

### Top Kmers Annotation Summary and Plotting ###
if top_kmers_go:
    logger.log(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Extracted top k-mers from {len(top_kmers_df['folder'].unique())} files.")
    logger.log(f"Value counts for 'organism' column:\n{top_kmers_df['organism'].value_counts()}")

    # Split by "entity" column 
    bact_kmers_df = top_kmers_df[top_kmers_df['organism'] == 'bacterium']
    phage_kmers_df = top_kmers_df[top_kmers_df['organism'] == 'phage']
    bact_len_before = len(bact_kmers_df)
    phage_len_before = len(phage_kmers_df)
    logger.log(f"Bacterium k-mers sample:\n{bact_kmers_df.head()}")
    logger.log(f"Phage k-mers sample:\n{phage_kmers_df.head()}")

    sort_by = 'UPS' if not weight_pfi else 'avg_expected_interaction'

    # Keep only top_kmers number of kkmers per entity per folder based on UPS score
    if not bact_kmers_df.empty:
        bact_kmers_df = balanced_top_k(
            df=bact_kmers_df,
            group_cols=['folder', 'entity'],
            sort_col=sort_by,
            total_k=top_kmers
        )
        logger.log(f"Top k-mers with {sort_by} scores - bacterium:")
        logger.log(f"{bact_kmers_df[['entity', 'decoded_kmer', sort_by]].head()}")
        bact_len_after = len(bact_kmers_df)

    else:
        logger.log(f"No valid bacterium k-mers data found for {sort_by} sorting.")
        bact_len_after = 0

    if not phage_kmers_df.empty:
        phage_kmers_df = balanced_top_k(
            df=phage_kmers_df,
            group_cols=['folder', 'entity'],
            sort_col=sort_by,
            total_k=top_kmers
        )
        logger.log(f"Top k-mers with {sort_by} scores - phage:")
        logger.log(f"{phage_kmers_df[['entity', 'decoded_kmer', sort_by]].head()}")
        phage_len_after = len(phage_kmers_df)
    else:
        logger.log(f"No valid phage k-mers data found for {sort_by} sorting.")
        phage_len_after = 0

    # Obtain pfi scores for kmers and add them to the dataframes if weight_pfi flag is set, then sort by pfi scores instead of UPS scores
    if weight_pfi:
        pass
    
    logger.log(f"Reduced bacterium k-mers from {bact_len_before} to {bact_len_after} based on top_kmers and sorting criteria.")
    logger.log(f"Reduced phage k-mers from {phage_len_before} to {phage_len_after} based on top_kmers and sorting criteria.")
    if bact_len_after > 0:
        logger.log(f"Final bacterium sample:\n{bact_kmers_df.head()}")
    if phage_len_after > 0:
        logger.log(f"Final phage sample:\n{phage_kmers_df.head()}")

    #Mutate UPS column to a number between 0 and 1 
    if 'UPS' in bact_kmers_df.columns:
        try:
            bact_kmers_df['UPS'] = bact_kmers_df['UPS'].apply(lambda x: np.random.random() if pd.notnull(x) else x)
            if (bact_kmers_df['UPS'] > 1).any():
                logger.log("Warning: 'UPS' column in bacterium k-mers dataframe contains values greater than 1. This may indicate an issue with the UPS calculation or data extraction.")
        except Exception as e:
            logger.log(f"Error converting 'UPS' column to numeric in bacterium k-mers dataframe: {e}")

    if 'UPS' in phage_kmers_df.columns:
        try:
            phage_kmers_df['UPS'] = phage_kmers_df['UPS'].apply(lambda x: np.random.random() if pd.notnull(x) else x)
            if (phage_kmers_df['UPS'] > 1).any():
                logger.log("Warning: 'UPS' column in phage k-mers dataframe contains values greater than 1. This may indicate an issue with the UPS calculation or data extraction.")
        except Exception as e:
            logger.log(f"Error converting 'UPS' column to numeric in phage k-mers dataframe: {e}")

    #return # for testing purposes, to check the outputs up to this point before proceeding with annotation and plotting
    # Gene analysis
    try:
        GA = GeneAnalysis()
        if not bact_kmers_df.empty:
            #bact_kmers_df = bact_kmers_df.reset_index()
            bact_annot_df = GA.batch_bact_annotate(bact_df=bact_kmers_df, kmer_col='decoded_kmer', entity_col='entity', data_prod_path=data_prod_path)
            #bact_annot_df = GA.batch_bact_annotate(bkmers=bact_kmers_df['decoded_kmer'].tolist(), bact_names=bact_kmers_df['entity'].tolist(), data_prod_path=data_prod_path)
        else:
            logger.log("No valid bacterium k-mers data found for annotation.")

        if not phage_kmers_df.empty:
            phage_annot_df = GA.batch_phage_annotate(phage_df=phage_kmers_df, kmer_col='decoded_kmer', entity_col='entity', data_prod_path=data_prod_path)
            #phage_annot_df = GA.batch_phage_annotate(pkmers=phage_kmers_df['decoded_kmer'].tolist(), phage_names=phage_kmers_df['entity'].tolist(), data_prod_path=data_prod_path)
        else:
            logger.log("No valid phage k-mers data found for annotation.")
    except Exception as e:
        raise ValueError(f"Error during gene annotation: {e}")


Scanning directory: /net/node07/home/projects/s215045/PredictPhagePPI/nn_runs/iterExcl_encoded_sketches_n500_k12
Output directory already exists: /net/node07/home/projects/s215045/PredictPhagePPI/data_prod//GAplot_test_n500_k12/
[2026-05-21 19:21:57]  collect_iterres started. Scanning /net/node07/home/projects/s215045/PredictPhagePPI/nn_runs/iterExcl_encoded_sketches_n500_k12 for log files.


Processing folders:   0%|          | 0/137 [00:00<?, ?it/s]


[2026-05-21 19:21:57] Processing folder: cluster_b0_p0_run1
Loaded hk_lookup JSON file from /net/node07/home/projects/s215045/PredictPhagePPI/data_prod/encoded_sketches/hk_lookup_n500_k12.json
Deduced hk_lookup from log info for cluster_b0_p0_run1 using path: /net/node07/home/projects/s215045/PredictPhagePPI/data_prod/encoded_sketches/hk_lookup_n500_k12.json
Found top kmers file: top_interaction_pair_kmers.csv in folder: cluster_b0_p0_run1
Skipping PFI calculation for cluster_b0_p0_run1. Reason: top_kmers=True, hk_lookup=True

[2026-05-21 19:21:57] Folder processed: cluster_b0_p0_run1


Processing folders:   1%|          | 1/137 [00:00<00:28,  4.78it/s]

##################################################

[2026-05-21 19:21:57] Processing folder: cluster_b0_p1_run1
Loaded hk_lookup JSON file from /net/node07/home/projects/s215045/PredictPhagePPI/data_prod/encoded_sketches/hk_lookup_n500_k12.json
Deduced hk_lookup from log info for cluster_b0_p1_run1 using path: /net/node07/home/projects/s215045/PredictPhagePPI/data_prod/encoded_sketches/hk_lookup_n500_k12.json
Found top kmers file: top_interaction_pair_kmers.csv in folder: cluster_b0_p1_run1


...

Deduced hk_lookup from log info for cluster_b0_p11_run1 using path: /net/node07/home/projects/s215045/PredictPhagePPI/data_prod/encoded_sketches/hk_lookup_n500_k12.json
Found top kmers file: top_interaction_pair_kmers.csv in folder: cluster_b0_p11_run1
Skipping PFI calculation for cluster_b0_p11_run1. Reason: top_kmers=True, hk_lookup=True

[2026-05-21 19:21:58] Folder processed: cluster_b0_p11_run1
##################################################

[2026-05-21 19:21:58] Processing folder: cluster_b0_p12_run1
Loaded hk_lookup JSON file from /net/node07/home/projects/s215045/PredictPhagePPI/data_prod/encoded_sketches/hk_lookup_n500_k12.json


Processing folders:   3%|▎         | 4/137 [00:01<00:41,  3.24it/s]

Deduced hk_lookup from log info for cluster_b0_p12_run1 using path: /net/node07/home/projects/s215045/PredictPhagePPI/data_prod/encoded_sketches/hk_lookup_n500_k12.json
Found top kmers file: top_interaction_pair_kmers.csv in folder: cluster_b0_p12_run1
Skipping PFI calculation for cluster_b0_p12_run1. Reason: top_kmers=True, hk_lookup=True

[2026-05-21 19:21:58] Folder processed: cluster_b0_p12_run1
##################################################
[2026-05-21 19:21:58] Extracted metrics from 5 log files.


Unable to process group_x_col: None, error: 'NoneType' object has no attribute 'lower'. Defaulting to no grouping.
⚠ WARNING: Some runs have failed! Count: 2
   feature_index  entity   organism  decoded_kmer              folder     UPS
0              2  J99_22  bacterium  GTTGTAGTACCA  cluster_b0_p0_run1  0.5527
1             13  J99_22  bacterium  TACGCGGACGTT  cluster_b0_p0_run1  0.5527
2             42  J99_22  bacterium  CCATAACGAGGA  cluster_b0_p0_run1  0.5527
3             46  J98_22  bacterium  ATCGTGGCGTTC  cluster_b0_p0_run1  0.5527
4             62  J99_22  bacterium  GGCGGTCTTCGA  cluster_b0_p0_run1  0.5527
Subsetted dataframe to 3 successful runs for plotting. Also subsetted top_kmers_df to 399 entries corresponding to successful runs.
Failed runs and their corresponding b_value and p_value:
  cluster_b0_p10_run1: b=0, p=10
  cluster_b0_p11_run1: b=0, p=11
All runs have the same 'n' & 'k' value. Results will be plotted based on rows and not grouped by 'n' and 'k'.
Using x_c

Annotating phage-kmer pairs: 100%|██████████| 101/101 [00:07<00:00, 12.75it/s]


## Investigating top_kmers_df

In [3]:
top_kmers_df

,feature_index,entity,organism,decoded_kmer,folder,UPS
0,2,J99_22,bacterium,GTTGTAGTACCA,cluster_b0_p0_run1,0.5527
1,13,J99_22,bacterium,TACGCGGACGTT,cluster_b0_p0_run1,0.5527
2,42,J99_22,bacterium,CCATAACGAGGA,cluster_b0_p0_run1,0.5527
3,46,J98_22,bacterium,ATCGTGGCGTTC,cluster_b0_p0_run1,0.5527
4,62,J99_22,bacterium,GGCGGTCTTCGA,cluster_b0_p0_run1,0.5527
...,...,...,...,...,...,...
794,2029,J99_22,bacterium,AATGTTGACCAG,cluster_b0_p12_run1,0.4414
795,5372,Ymer,phage,ATTATCAGCGAC,cluster_b0_p12_run1,0.4414
796,9006,Poppous,phage,GATGTAGGGCCT,cluster_b0_p12_run1,0.4414
797,9766,Amona,phage,TATTTTCGGTGG,cluster_b0_p12_run1,0.4414


In [4]:
bact_kmers_df

,feature_index,entity,organism,decoded_kmer,folder,UPS
178,1797,J99_22,bacterium,GACATATGACTG,cluster_b0_p1_run1,0.282711
179,1868,J99_22,bacterium,TATAAGCATCCA,cluster_b0_p1_run1,0.682210
180,1878,J98_22,bacterium,CCGAGTTGGGCC,cluster_b0_p1_run1,0.754429
181,1879,J99_22,bacterium,CTGTCGCTACCC,cluster_b0_p1_run1,0.342979
182,1956,J99_22,bacterium,GGAAAGGATTTT,cluster_b0_p1_run1,0.545012
...,...,...,...,...,...,...
767,1480,J99_22,bacterium,TTATCCGCCTCT,cluster_b0_p12_run1,0.775690
768,1483,J99_22,bacterium,TATGGGAATGGG,cluster_b0_p12_run1,0.759482
769,1495,J99_22,bacterium,AACTCATCTCTC,cluster_b0_p12_run1,0.796970
770,1563,J99_22,bacterium,CCCACCGAACAC,cluster_b0_p12_run1,0.184338


In [5]:
bact_annot_df.sort_values(by='UPS', ascending=False)

,bact,locus_tag,kmer_in_seq,length_bp,gene,product,UPS
175,J98_22,OJIANMJK_02853,GAACGCATACAT,1113,fdtB,"dTDP-3-amino-3,6-dideoxy-alpha-D-galactopyrano...",0.948903
174,J98_22,OJIANMJK_02853,GAACGCATACAT,1113,fdtB,NaN,0.948903
137,J99_22,KCDMCKIA_03201,TGGACGCGGCGG,1122,ispG,4-hydroxy-3-methylbut-2-en-1-yl diphosphate sy...,0.924234
136,J99_22,KCDMCKIA_03201,TGGACGCGGCGG,1122,ispG,NaN,0.924234
135,J99_22,KCDMCKIA_00986,TGGACGCGGCGG,1344,hipO,Hippurate hydrolase,0.924234
...,...,...,...,...,...,...,...
99,J99_22,KCDMCKIA_01600,CCTGAATCATCA,978,NaN,hypothetical protein,0.010594
100,J99_22,KCDMCKIA_03711,CCTGAATCATCA,1194,estB,NaN,0.010594
101,J99_22,KCDMCKIA_03711,CCTGAATCATCA,1194,estB,Esterase EstB,0.010594
96,J99_22,KCDMCKIA_01293,CCTGAATCATCA,657,pxpB,NaN,0.010594


In [6]:
top_k = bact_annot_df["kmer_in_seq"].value_counts().head(top_kmers).index.tolist()
top_k_set = set(top_k)

kmer_scores = (
    bact_annot_df[bact_annot_df["kmer_in_seq"].isin(top_k_set)]
    .groupby("kmer_in_seq")["UPS"]
    .mean()
)
ordered_kmers = sorted(
    (k for k in kmer_scores.index if k in top_k_set),
    key=lambda k: -kmer_scores.loc[k],
)

print(ordered_kmers)

['TGATGACTTGCG', 'CTGAATACGTAC', 'GAACGCATACAT', 'CAATCCCGCGCC', 'CGAAACGGGTTA', 'CCGGGATCGACG', 'TGGACGCGGCGG', 'CGCTGCAGAATG', 'ATTAAAGCCGTA', 'GCGGCAAAGATC', 'ACCAGCGGCCAG', 'CGCCAGCGCCGA', 'TCGATGACGCCG', 'GTACGGCTTGAA', 'TTTTGTGCGTTA', 'CAACAGCGCACA', 'TCAGGCTCGCGA', 'TACCAGTTACAC', 'GGAAAGGATTTT', 'CGGCTTTCGCCG', 'CTGTCGCTACCC', 'GACTGGAACAGG', 'GGTTGTGGCATC', 'AACCTGACCGTT', 'CTGGCAGCATTT', 'ACTACAGGCAGG', 'TGGCGGCATCAG', 'TATCTTACAAGG', 'AATGTTGACCAG', 'GTCCAATACTAC', 'AGCACCATGACG', 'TCTATTCATTAT', 'GCGCCTGATGAG', 'GCCAGGCTTCTC', 'AAGTGGCTTCTC', 'ATCGTGGCGTTC', 'AAGCGTTGATGG', 'ACTGATGCAGGA', 'CGCTCAGGACAT', 'CCTGAATCATCA']


Check if UPS column is in descending order

In [7]:
ups = bact_annot_df["UPS"].dropna().astype(float).values
if ups.size <= 1:
    print("UPS has <=1 non-NaN values; considered descending=True")
else:
    is_desc = np.all(ups[:-1] >= ups[1:])
    print(f"is_descending: {is_desc}")
    if not is_desc:
        viol_idx = np.where(ups[:-1] < ups[1:])[0][0]
        print(f"First violation at index {viol_idx} -> {viol_idx+1}: {ups[viol_idx]} < {ups[viol_idx+1]}")

is_descending: False
First violation at index 1 -> 2: 0.3429794681373022 < 0.5450122156045017


In [11]:
def _normalize_for_combine(df, species_col, entity_col, organism_label):
    """Rename species/entity-label columns to a shared schema, add organism tag."""
    out = df.copy()
    out['species'] = out[species_col]
    out['entity_label'] = out[entity_col]   # gene name for bact, product for phage
    out['organism'] = organism_label
    return out

# Gene Annot Plotting
try:
    title_suffix = "(PFI)" if weight_pfi else "(UPS)"
    plotting_utils = GAPlottingUtils(df=top_kmers_df, outdir=str(outdir))
    if not bact_kmers_df.empty:
        plotting_utils.plot_top_genes(bact_annot_df, entity_type="bacterium", title_suffix=title_suffix)
        plotting_utils.plot_kmer_distribution(bact_annot_df, entity_type="bacterium", title_suffix=title_suffix)
        plotting_utils.plot_kmer_gene_network(bact_annot_df, entity_type="bacterium", top_kmers=network_top_kmers)
        if weight_pfi:
            plotting_utils.plot_kmer_against_ups_or_pfi(bact_annot_df, entity_type="bacterium", sort_by=sort_by)

    if not phage_kmers_df.empty:
        plotting_utils.plot_top_genes(phage_annot_df, entity_type="phage", title_suffix=title_suffix)
        plotting_utils.plot_kmer_distribution(phage_annot_df, entity_type="phage", title_suffix=title_suffix)
        plotting_utils.plot_kmer_gene_network(phage_annot_df, entity_type="phage", top_kmers=network_top_kmers)
        if weight_pfi:
            plotting_utils.plot_kmer_against_ups_or_pfi(phage_annot_df, entity_type="phage", sort_by=sort_by)
            
except Exception as e:
    raise ValueError(f"Error during gene annotation plotting: {e}")

### Combined plotting
frames = []
if not bact_kmers_df.empty and 'bact_annot_df' in locals():
    frames.append(_normalize_for_combine(
        bact_annot_df, species_col='bact', entity_col='gene',
        organism_label='bacterium'))
if not phage_kmers_df.empty and 'phage_annot_df' in locals():
    frames.append(_normalize_for_combine(
        phage_annot_df, species_col='contig_id', entity_col='product',
        organism_label='phage'))

if frames:
    collected_df = pd.concat(frames, ignore_index=True, sort=False)
    collected_df.to_csv(outdir + 'top_kmers_annotations.csv', index=False)
    logger.log(f"Combined annotation frame: {len(collected_df)} rows, "
            f"{collected_df['species'].nunique()} species, "
            f"{collected_df['organism'].nunique()} organism types.")
else:
    collected_df = pd.DataFrame()
    logger.log("No annotated rows to combine.")

try:
    if not collected_df.empty:
        plotting_utils.plot_species_distribution_grid(
            collected_df, value_col='decoded_kmer', top_n=30)
        plotting_utils.plot_species_distribution_grid(
            collected_df, value_col='entity_label', top_n=30)
except Exception as e:
    raise ValueError(f"Error during combined annotation plotting: {e}")

# Concatenate annotation results and save
try: 
    if not bact_kmers_df.empty and not phage_kmers_df.empty:
        combined_annot_df = pd.concat([bact_annot_df, phage_annot_df], ignore_index=True)
        combined_annot_df.to_csv(outdir + 'top_kmers_annotations.csv', index=False)
        logger.log("Top Kmers Annotations CSV saved as top_kmers_annotations.csv")
    
    elif not bact_kmers_df.empty:
        bact_annot_df.to_csv(outdir + 'top_kmers_annotations.csv', index=False)
        logger.log("Bacterium Kmers Annotations CSV saved as top_kmers_annotations.csv")
    elif not phage_kmers_df.empty:
        phage_annot_df.to_csv(outdir + 'top_kmers_annotations.csv', index=False)
        logger.log("Phage Kmers Annotations CSV saved as top_kmers_annotations.csv")
    
    # Additionally: build a mapping of decoded_kmer -> annotated genes/products
    try:
        mapping_frames = []
        # Bacterium mapping: 'decoded_kmer' -> 'gene'
        if not bact_kmers_df.empty and 'bact_annot_df' in locals():
            if 'decoded_kmer' in bact_annot_df.columns and 'gene' in bact_annot_df.columns:
                df_bmap = bact_annot_df[['decoded_kmer', 'gene']].dropna()
                if not df_bmap.empty:
                    df_bmap = df_bmap.groupby('decoded_kmer')['gene'].agg(lambda x: ';'.join(sorted(set(x)))).reset_index()
                    df_bmap = df_bmap.rename(columns={'gene': 'mapped_genes'})
                    df_bmap['organism'] = 'bacterium'
                    mapping_frames.append(df_bmap)

        # Phage mapping: 'decoded_kmer' -> 'product'
        if not phage_kmers_df.empty and 'phage_annot_df' in locals():
            if 'decoded_kmer' in phage_annot_df.columns and 'product' in phage_annot_df.columns:
                df_pmap = phage_annot_df[['decoded_kmer', 'product']].dropna()
                if not df_pmap.empty:
                    df_pmap = df_pmap.groupby('decoded_kmer')['product'].agg(lambda x: ';'.join(sorted(set(x)))).reset_index()
                    df_pmap = df_pmap.rename(columns={'product': 'mapped_genes'})
                    df_pmap['organism'] = 'phage'
                    mapping_frames.append(df_pmap)

        if mapping_frames:
            kmer_map_df = pd.concat(mapping_frames, ignore_index=True, sort=False)
            # Count how many distinct genes/products each kmer maps to
            kmer_map_df['num_genes'] = kmer_map_df['mapped_genes'].apply(lambda s: 0 if pd.isna(s) or s == '' else len(str(s).split(';')))
            # Save mapping CSV for user inspection
            kmer_map_df.to_csv(outdir + 'kmer_to_genes_mapping.csv', index=False)
            logger.log("Saved kmer-to-genes mapping CSV: kmer_to_genes_mapping.csv")

            # Write a short human-readable summary with examples of multi-mapping kmers
            total_kmers = len(kmer_map_df)
            multi_map_count = int((kmer_map_df['num_genes'] > 1).sum())
            examples = kmer_map_df[kmer_map_df['num_genes'] > 1].head(20)
            summary_lines = [f"Total unique annotated kmers: {total_kmers}", f"Kmers mapping to multiple genes/products: {multi_map_count}", "\nExamples of kmers mapping to multiple genes/products (up to 20):"]
            for _, r in examples.iterrows():
                summary_lines.append(f"{r.get('decoded_kmer','<unknown kmer>')} ({r['organism']}): {r['mapped_genes']}")
            with open(outdir + 'top_kmers_mapping_summary.txt', 'w') as summary_f:
                summary_f.write('\n'.join(summary_lines))
            logger.log("Saved human-readable mapping summary: top_kmers_mapping_summary.txt")

            # Log concise counts for immediate visibility
            logger.log(f"Kmer mapping summary: {total_kmers} unique kmers; {multi_map_count} map to multiple genes/products.")

            # (Agent-only TODO tracking completed outside of runtime)

    except Exception as e:
        logger.log(f"Warning: Failed to build/save kmer->genes mapping: {e}")
except Exception as e:
    raise ValueError(f"Error saving top k-mers annotations: {e}")

Ordering kmers by UPS (descending, aggregated by mean).
Removed 5 isolated nodes from the network.
Ordering kmers by UPS (descending, aggregated by mean).
Combined annotation frame: 187 rows, 5 species, 2 organism types.
plot_species_distribution_grid: missing data for decoded_kmer.
Top Kmers Annotations CSV saved as top_kmers_annotations.csv


In [ ]:
phage_kmers_df

,feature_index,entity,organism,decoded_kmer,folder,UPS
183,2780,Poppous,phage,TCGGTTCATCTC,cluster_b0_p1_run1,0.129378
185,2914,Poppous,phage,CTGCGAATAGTG,cluster_b0_p1_run1,0.013491
184,2857,Amona,phage,TAAATGATCTAA,cluster_b0_p1_run1,0.826142
187,3181,Poppous,phage,ATCGGGCAGCTA,cluster_b0_p1_run1,0.131650
186,3020,Poppous,phage,TTTGCAATACGC,cluster_b0_p1_run1,0.499454
...,...,...,...,...,...,...
145,8060,Amona,phage,CGCCTGGAATTT,cluster_b0_p0_run1,0.914013
795,5372,Ymer,phage,ATTATCAGCGAC,cluster_b0_p12_run1,0.649610
796,9006,Poppous,phage,GATGTAGGGCCT,cluster_b0_p12_run1,0.713620
797,9766,Amona,phage,TATTTTCGGTGG,cluster_b0_p12_run1,0.803274
